# Professor Notebooks 01-06 Combined

교수님이 제공한 `notebook_01.py`부터 `notebook_06.py`까지를 빠짐없이 순서대로 실행하기 위한 통합 Colab 노트북입니다.

흐름: Bigram → MLP names → MLP Shakespeare → GPT-style dataset → Single-head masked self-attention → TinyGPT.


# Notebook 01
Original file: `notebook_01.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_01.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1nayKBU-7eJ1BhdNiPspB-FmXPxnxheaP

# Notebook 1 — Bigram Language Model on `names.txt`

이 노트북의 목표는 **가장 기본적인 문자 단위 language model**을 만드는 것입니다.

- 입력: 현재 문자 1개
- 출력: 다음 문자 1개
- 즉, **bigram** 확률표를 학습합니다.

전체 구조는 `Dataset → DataLoader → Model → Loss → Train loop → Sampling`입니다.

## 1. 데이터 준비
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
chars = ['.'] + chars

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
encoded_words = [[stoi[ch] for ch in w] for w in words]

print("num words:", len(words))
print("vocab_size:", vocab_size)
print(words[:10])

"""## 2. Dataset (`block_size = 1`)"""

class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X, self.Y = [], []
        for word in encoded_words:
            context = [0] * block_size
            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)
                context = context[1:] + [ix]
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

block_size = 1
dataset = NamesContextDataset(encoded_words, block_size)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

x, y = dataset[1]
xb, yb = next(iter(loader))
print("single x:", x, x.shape)
print("single y:", y)
print("batch xb.shape:", xb.shape)
print("batch yb.shape:", yb.shape)

"""## 3. Bigram model (명시적 one-hot 버전)"""

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.W = nn.Parameter(torch.randn(vocab_size, vocab_size))

    def forward(self, x):
        x = x.view(-1)  # (B,)
        x_onehot = F.one_hot(x, num_classes=self.vocab_size).float()  # (B, V)
        logits = x_onehot @ self.W                                    # (B, V)
        return logits

model = BigramLanguageModel(vocab_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

# @title
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x):
        x = x.view(-1)  # (B,)
        logits = self.token_embedding_table(x)  # (B, V)
        return logits

"""## 4. Train / Eval 함수"""

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

"""## 5. 학습"""

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BigramLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-1)

for epoch in range(20):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 6. Sampling"""

@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval()
    results = []
    for _ in range(num_samples):
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = []
        for _ in range(max_len):
            logits = model(context)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()
            if next_token == 0:
                break
            out.append(itos[next_token])
            context = torch.cat([context[:, 1:], ix], dim=1)
        results.append("".join(out))
    return results

sample(model, block_size, itos, device, num_samples=10)

"""## 7. 정리

- `block_size=1`이면 bigram 문제를 정확히 표현할 수 있습니다.
- one-hot과 행렬곱만으로도 language model을 만들 수 있습니다.
- 이 학습 루프는 이후 단계에서도 거의 그대로 재사용됩니다.
"""



# Notebook 02
Original file: `notebook_02.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_02.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1JtteLHSjBGkv7IJgBhmaGV4F64rNxtG7

# Notebook 2 — MLP Character Model on `names.txt`

이제 `names.txt`는 그대로 두고, **dataset을 일반화**하고 **모델을 MLP로 확장**합니다.

- bigram: 현재 문자 1개만 사용
- MLP: 길이 `block_size`의 context 전체 사용
- 표현도 one-hot 대신 **embedding**을 사용
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
chars = ['.'] + chars
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
encoded_words = [[stoi[ch] for ch in w] for w in words]

"""## 1. 일반화된 context dataset"""

class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X, self.Y = [], []
        for word in encoded_words:
            context = [0] * block_size
            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)
                context = context[1:] + [ix]
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

block_size = 3
dataset = NamesContextDataset(encoded_words, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

"""## 2. MLP model"""

class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=10, hidden_dim=200):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

"""## 3. Train / Eval"""

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

"""## 4. 학습"""

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 5. Sampling"""

@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval()
    results = []
    for _ in range(num_samples):
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = []
        for _ in range(max_len):
            logits = model(context)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()
            if next_token == 0:
                break
            out.append(itos[next_token])
            context = torch.cat([context[:, 1:], ix], dim=1)
        results.append("".join(out))
    return results

sample(model, block_size, itos, device, num_samples=10)

"""## 6. 정리

- dataset은 `context -> next char` 구조를 유지합니다.
- 차이는 모델 내부입니다.
- embedding과 MLP를 사용하므로 bigram보다 더 풍부한 문맥을 처리할 수 있습니다.
"""



# Notebook 03
Original file: `notebook_03.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_03.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ZrENCG_0g7bf3efpthSFqEEqjAlElBJC

# Notebook 3 — MLP on Tiny Shakespeare

이제 모델은 2번 MLP를 그대로 두고, **데이터만 `tiny Shakespeare`로 바꿉니다.**

문제는 여전히 **fixed context -> next char** 입니다.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab_size:", vocab_size)

print(text)

"""## 1. Sliding-window dataset"""

class CharSequenceNextCharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + self.block_size]
        return x, y

block_size = 16
dataset = CharSequenceNextCharDataset(data, block_size)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)
print("decoded x:", ''.join(itos[i.item()] for i in xb[0]))
print("decoded y:", itos[yb[0].item()])

"""## 2. MLP model"""

class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

"""## 3. 학습"""

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(10):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 4. Sampling"""

@torch.no_grad()
def sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = [0] * block_size
    for ch in start_text:
        if ch in stoi:
            context = context[1:] + [stoi[ch]]
    out = list(start_text)
    for _ in range(max_new_tokens):
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1).item()
        out.append(itos[ix])
        context = context[1:] + [ix]
    return "".join(out)

print(sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

"""## 5. 정리

- 같은 MLP 모델을 더 큰 텍스트에도 적용할 수 있습니다.
- 바뀌는 것은 dataset입니다.
- 하지만 fixed context MLP는 긴 문맥을 잘 다루지 못합니다.
"""



# Notebook 04
Original file: `notebook_04.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_04.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1D82L3exi2d4Pr4ffOBCGICW4FDqNf_hb

# Notebook 4 — GPT-style Dataset + Minimal Sequence Model

이제부터는 **target 구조 자체**가 바뀝니다.

기존에는 `y`가 다음 문자 1개였지만, 이제는 `y`도 sequence입니다.

- `x = [t1, t2, ..., tT]`
- `y = [t2, t3, ..., t(T+1)]`
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print(text)

"""## 1. GPT-style dataset"""

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb[0]

"""## 2. Attention 없는 최소 sequence model"""

class TinySequenceLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)            # (B, T, C)
        pos = self.position_embedding(pos)[None] # (1, T, C)
        h = tok + pos
        logits = self.lm_head(h)                 # (B, T, V)
        return logits

model = TinySequenceLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

"""## 3. Loss"""

def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

print("initial loss:", sequence_cross_entropy(logits, yb).item())

"""## 4. 학습"""

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinySequenceLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 5. Sampling"""

@torch.no_grad()
def sample_sequence_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_sequence_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

"""## 6. 정리

- 이제 target도 sequence입니다.
- output shape는 `(B, T, V)`입니다.
- positional embedding이 처음 도입됩니다.
- 아직 attention은 없지만 GPT의 데이터/출력 인터페이스는 이미 갖추었습니다.
"""


# Notebook 05
Original file: `notebook_05.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_05.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1KL-ilWg8EsCrWG8FXwnGrjNQH2k3A-j6

# Notebook 5 — Single-Head Masked Self-Attention

이제 GPT의 핵심 아이디어인 **masked self-attention**을 추가합니다.

이번 단계에서는 **single-head**에만 집중합니다.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

"""## 1. Single-head masked self-attention"""

class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

"""## 2. Attention 포함 최소 모델"""

class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.attn(h)
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

"""## 3. 학습"""

def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyAttentionLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 4. Sampling"""

@torch.no_grad()
def sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

"""## 5. 정리

- 각 위치는 이전 위치들을 참조할 수 있습니다.
- 미래는 causal mask로 차단됩니다.
- 이제 모델이 어떤 위치를 참고할지 스스로 결정하기 시작합니다.
"""



# Notebook 06
Original file: `notebook_06.py`


In [ ]:
# -*- coding: utf-8 -*-
"""notebook_06.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1BHD1I9Xh4iV_PX8t2DZxRVjhUBxC82kk

# Notebook 6 — Toward a Tiny GPT

이제 남은 모듈들을 하나씩 추가하여 **좀 더 GPT다운 구조**로 갑니다.

이번 노트북에서 추가하는 것들:

- multi-head attention
- feedforward network
- residual connection
- layer normalization
- block stacking
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 64
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

"""## 1. Multi-head attention"""

class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

"""## 2. Feedforward + Block"""

class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

"""## 3. Tiny GPT"""

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

"""## 4. 학습"""

def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

"""## 5. Sampling"""

@torch.no_grad()
def sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=500))

"""## 6. 정리

이제 아주 작은 GPT 구조가 완성되었습니다.

전체 흐름은 다음과 같습니다.

1. bigram
2. MLP on names
3. MLP on Shakespeare
4. GPT-style dataset + minimal sequence model
5. single-head masked self-attention
6. tiny GPT
"""



# 제출용 캡처 체크

- Notebook 1: Bigram loss와 sampling 결과
- Notebook 2: MLP names loss와 sampling 결과
- Notebook 3: Tiny Shakespeare MLP loss와 ROMEO 샘플
- Notebook 4: `xb.shape`, `yb.shape`, `logits.shape`, sequence loss
- Notebook 5: masked self-attention 포함 모델 loss와 샘플
- Notebook 6: TinyGPT loss와 최종 샘플
